<h1 style="
    color:#FFFFFF;
    background:linear-gradient(90deg,#0F766E,#0284C7);
    text-align:center;
    font-weight:bold;
    padding:18px 12px;
    border-radius:10px;
    margin-bottom:7px;">
    ScaleFlow — Prepare and Preprocess the AI/ML Dataset
</h1>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">
    Apache JIRA Data → Clean, Documented, ML-Ready ScaleFlow Dataset
</h3>

<div style="border-left:6px solid #FB7185;background-color:#FFF1F2;padding:12px 16px;margin:16px 0;border-radius:6px;color:#7F1D1D;">
<b>ScaleFlow • AI/ML • Data Preparation</b><br>
Branch: <code>sc-prepare-and-preprocess-the-AI-ML-dataset</code><br>
Structured raw sources: <b>issues</b>, <b>issue links</b>, and <b>changelog</b>.<br>
The <b>comments</b> archive is kept for future NLP / Generative AI enrichment.
</div>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p><b>Main question:</b></p>
<div style="font-size:1.12em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:16px;border-radius:8px;margin:14px 0;color:#111827;">
How can we transform the full Apache JIRA dataset into a clean, reproducible, leakage-aware dataset for ScaleFlow's first AI/ML models?
</div>
<p style="margin-bottom:0;"><b>Notebook style:</b> the same inline HTML/CSS visual language used in Week 9 — Day 2, with simple pandas code that is easy to read, run, and explain.</p>
</div>

<a id="toc"></a>
<h2 style="color:#92400E;background-color:#FBBF24;font-weight:bold;margin-top:30px;padding:9px 14px;border-radius:7px;">Table of Contents</h2>
<div style="border:1px solid #CBD5E1;padding:16px 25px;border-radius:8px;background-color:#FFFFFF;color:#111827;">
<ul style="font-weight:bold;line-height:1.95;color:#1F2937;">
<li><a href="#section0">0. Setup — Files, Imports & Environment</a></li>
<li><a href="#section1">1. Learning Map & Study Basis</a></li>
<li><a href="#section2">2. Dataset Roles & ScaleFlow Mapping</a></li>
<li><a href="#section3">3. Inspect the Raw Dataset</a></li>
<li><a href="#section4">4. Clean and Prepare the Issues Dataset</a></li>
<li><a href="#section5">5. Build Dependency Features from Issue Links</a></li>
<li><a href="#section6">6. Build Historical Features from Changelog</a></li>
<li><a href="#section7">7. Merge the JIRA Sources</a></li>
<li><a href="#section8">8. Feature Engineering, Targets & Leakage Control</a></li>
<li><a href="#section9">9. Create Time-Based Train / Validation / Test Splits</a></li>
<li><a href="#section10">10. Encode Categorical Features</a></li>
<li><a href="#section11">11. Export Final Dataset & ML-Ready Splits</a></li>
<li><a href="#section12">12. Data Quality Checks</a></li>
<li><a href="#section13">13. Task Definition of Done</a></li>
</ul>
</div>


<a id="section0"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">0. Setup — Files, Imports & Environment</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">0.1 Expected project structure</h3>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">ScaleFlow/
└── ScaleFlow/
    └── AI_ML/
        ├── Data/
        │   ├── raw/
        │   │   ├── issues.zip
        │   │   ├── issuelinks.zip
        │   │   ├── changelog.zip
        │   │   └── comments.csv.7z
        │   └── processed/
        │       ├── final_dataset.csv
        │       ├── train.csv
        │       ├── validation.csv
        │       ├── test.csv
        │       ├── jira_to_scaleflow_mapping.csv
        │       └── preprocessing_metadata.json
        └── Notebooks/
            └── ScaleFlow_AI_ML_Data_Preparation.ipynb</div>

<p>The raw files remain unchanged. Every generated dataset is written to <code>Data/processed</code>.</p>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> The raw JIRA files are large. This notebook uses pandas chunk processing so the full files can be processed without loading all raw rows into memory at once.</div>
</div>

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

CHUNK_SIZE = 100_000
INSPECTION_ROWS = 5

print("pandas version:", pd.__version__)
print("Chunk size:", f"{CHUNK_SIZE:,}")


pandas version: 2.3.3
Chunk size: 100,000


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Why this setup?</b> The notebook follows the same execute → inspect → explain workflow used in the BinX notebooks. The large files are processed in pieces, while small previews are used only for understanding the schema.</div>

In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "Notebooks":
    AI_ML_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "AI_ML").exists():
    AI_ML_DIR = CURRENT_DIR / "AI_ML"
elif (CURRENT_DIR / "ScaleFlow" / "AI_ML").exists():
    AI_ML_DIR = CURRENT_DIR / "ScaleFlow" / "AI_ML"
else:
    raise FileNotFoundError(
        "AI_ML folder was not found. Please run the notebook from the ScaleFlow project."
    )

RAW_DIR = AI_ML_DIR / "Data" / "raw"
PROCESSED_DIR = AI_ML_DIR / "Data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ISSUES_PATH = RAW_DIR / "issues.zip"
LINKS_PATH = RAW_DIR / "issuelinks.zip"
CHANGELOG_PATH = RAW_DIR / "changelog.zip"
COMMENTS_PATH = RAW_DIR / "comments.csv.7z"

CLEAN_ISSUES_PATH = PROCESSED_DIR / "_clean_issues.csv"
ENRICHED_PATH = PROCESSED_DIR / "_enriched_dataset.csv"
FINAL_DATASET_PATH = PROCESSED_DIR / "final_dataset.csv"
TRAIN_PATH = PROCESSED_DIR / "train.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
MAPPING_PATH = PROCESSED_DIR / "jira_to_scaleflow_mapping.csv"
METADATA_PATH = PROCESSED_DIR / "preprocessing_metadata.json"

print("Current working directory:", CURRENT_DIR)
print("AI_ML folder:", AI_ML_DIR)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)


Current working directory: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Notebooks
AI_ML folder: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML
Raw data folder: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Data\raw
Processed data folder: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Data\processed


In [3]:
raw_files = {
    "Issues": ISSUES_PATH,
    "Issue Links": LINKS_PATH,
    "Changelog": CHANGELOG_PATH,
    "Comments": COMMENTS_PATH,
}

inventory = []
for name, path in raw_files.items():
    inventory.append({
        "dataset": name,
        "file": path.name,
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else None,
    })

display(pd.DataFrame(inventory))

required_files = [ISSUES_PATH, LINKS_PATH, CHANGELOG_PATH]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing_files))

print("Required structured datasets are available.")


,dataset,file,exists,size_mb
0,Issues,issues.zip,True,375.32
1,Issue Links,issuelinks.zip,True,19.62
2,Changelog,changelog.zip,True,487.57
3,Comments,comments.csv.7z,True,493.01


Required structured datasets are available.


<a id="section1"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">1. Learning Map & Study Basis</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>This notebook intentionally uses the same learning style as Week 9 Day 2: a clear flow, short explanations before the code, small interpretation boxes after important stages, and a final Definition of Done.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Raw JIRA archives
      ↓
Inspect columns and data types
      ↓
Clean issues / links / changelog
      ↓
Aggregate relationships and history
      ↓
Merge on task key
      ↓
Feature engineering
      ↓
Prepare honest targets
      ↓
Control data leakage
      ↓
Chronological split
      ↓
Train / Validation / Test
      ↓
Final ScaleFlow ML-ready files</div>



<a id="section2"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">2. Dataset Roles & ScaleFlow Mapping</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">2.1 What each JIRA file contributes</h3>
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Raw file</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Role</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Used now?</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>issues.zip</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Main issue/task metadata and lifecycle timestamps.</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Yes</b></td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>issuelinks.zip</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Relationships between issues; used for link and blocking-dependency features.</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Yes</b></td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>changelog.zip</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Historical field changes; used for status, assignee, priority, issue-type, and project change counts.</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Yes</b></td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>comments.csv.7z</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Free-text discussion; useful later for NLP / Generative AI features.</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Not in the current structured MVP</b></td></tr>
</table>

<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> Adding the comments archive to the project does not mean it must be used in the first structured model. It remains available for future NLP or Generative AI enrichment.</div>
</div>

In [4]:
mapping_df = pd.DataFrame({
    "JIRA source": [
        "issues.csv", "issues.csv", "issues.csv", "issues.csv", "issues.csv",
        "issues.csv", "issues.csv", "issuelinks.csv", "issuelinks.csv",
        "changelog.csv", "changelog.csv", "changelog.csv", "comments.csv"
    ],
    "JIRA field": [
        "key", "project.key", "priority.name", "issuetype.name", "issuetype.subtask",
        "created", "resolutiondate", "all link rows", "blocking links",
        "status", "assignee", "priority", "comment text"
    ],
    "ScaleFlow field": [
        "task_key", "project_key", "priority", "issue_type", "is_subtask",
        "created_date", "completed_date", "issue_link_count", "dependency_count",
        "status_change_count", "assignee_change_count", "priority_change_count",
        "future_text_features"
    ],
    "Role": [
        "Identifier", "Feature", "Feature", "Feature", "Feature",
        "Feature / split", "Target source", "Analytical feature", "Analytical feature",
        "Historical feature", "Historical feature", "Leakage check", "Future work"
    ],
})

display(mapping_df)
mapping_df.to_csv(MAPPING_PATH, index=False)
print("Saved mapping:", MAPPING_PATH)


,JIRA source,JIRA field,ScaleFlow field,Role
0,issues.csv,key,task_key,Identifier
1,issues.csv,project.key,project_key,Feature
2,issues.csv,priority.name,priority,Feature
3,issues.csv,issuetype.name,issue_type,Feature
4,issues.csv,issuetype.subtask,is_subtask,Feature
5,issues.csv,created,created_date,Feature / split
6,issues.csv,resolutiondate,completed_date,Target source
7,issuelinks.csv,all link rows,issue_link_count,Analytical feature
8,issuelinks.csv,blocking links,dependency_count,Analytical feature
9,changelog.csv,status,status_change_count,Historical feature


Saved mapping: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Data\processed\jira_to_scaleflow_mapping.csv


<div style="border-left:5px solid #0284C7;background-color:#E0F2FE;padding:11px 15px;margin:12px 0;border-radius:5px;color:#0C4A6E;"><b>Target limitation:</b> The selected JIRA issues table does not provide a reliable planned due date for every issue. Therefore this notebook does not invent a fake <code>is_delayed</code> label. The first honest target is <code>resolution_time_days</code>.</div>

<a id="section3"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">3. Inspect the Raw Dataset</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">3.1 Why inspection comes first</h3>
<p>Before cleaning, we need to understand the available columns, inferred data types, missing values, and example rows. The preview is small only for inspection. The real preparation code later processes the full archives in chunks.</p>
</div>

In [5]:
issues_preview = pd.read_csv(
    ISSUES_PATH, compression="zip", nrows=INSPECTION_ROWS, low_memory=False
)
links_preview = pd.read_csv(
    LINKS_PATH, compression="zip", nrows=INSPECTION_ROWS, low_memory=False
)
changelog_preview = pd.read_csv(
    CHANGELOG_PATH, compression="zip", nrows=INSPECTION_ROWS, low_memory=False
)

for name, df in {
    "issues.csv": issues_preview,
    "issuelinks.csv": links_preview,
    "changelog.csv": changelog_preview,
}.items():
    print()
    print("=" * 80)
    print(name)
    print("Preview shape:", df.shape)
    display(df.head())



issues.csv
Preview shape: (5, 37)


,id,key,summary,resolution.id,resolution.description,resolution.name,priority.id,priority.name,labels,assignee,status.id,status.description,status.name,statusCategory.id,statusCategory.key,statusCategory.colorName,statusCategory.name,customfield_12310921,creator,subtasks,reporter,votes.votes,issuetype.id,issuetype.description,issuetype.name,issuetype.subtask,project.id,project.key,project.name,projectCategory.id,projectCategory.description,projectCategory.name,resolutiondate,watches.watchCount,created,updated,description
0,12451174,WW-712,Update config browser to work with the new syntax,1.0,A fix for this issue is checked into the tree ...,Fixed,4,Minor,[],632ec9dd,6,"The issue is considered finished, the resoluti...",Closed,3,done,green,Done,NaN,632ec9dd,[],632ec9dd,0,4,An improvement or enhancement to an existing f...,Improvement,False,12311041,WW,Struts 2,10380,struts.apache.org,Struts Framework,2005-01-01 07:50:46,0,2005-01-01 07:47:50,2005-01-01 07:50:46,The config browser used Velocity calling the t...
1,29159,XALANC-446,XALAN_C 1.9 or current do not build on Fedora ...,1.0,A fix for this issue is checked into the tree ...,Fixed,1,Blocker,[],NaN,5,"A resolution has been taken, and it is awaitin...",Resolved,3,done,green,Done,NaN,4fc0930d,[],4fc0930d,0,1,A problem which impairs or prevents the functi...,Bug,False,10582,XALANC,XalanC,11460,Xalan Related Projects,Xalan,2004-12-30 05:30:36,1,2004-12-25 22:50:30,2005-01-01 10:20:52,Two types of errors:\n1- runConfigure and conf...
2,12420130,ROL-587,"Problem with ADD new post, and DELETE post.",5.0,"All attempts at reproducing this issue failed,...",Cannot Reproduce,2,Critical,[],4d4054ac,6,"The issue is considered finished, the resoluti...",Closed,3,done,green,Done,NaN,110153ae,[],110153ae,0,1,A problem which impairs or prevents the functi...,Bug,False,12310906,ROL,Apache Roller,10331,NaN,Roller,2005-01-02 15:21:00,0,2005-01-01 13:52:46,2005-01-02 15:21:00,"When trying to add new post, I was getting nex..."
3,29222,AXIS-1741,LogHandler can only work in GlobalConfiguratio...,NaN,NaN,NaN,3,Major,[],NaN,1,The issue is open and ready for the assignee t...,Open,2,new,blue-gray,To Do,NaN,be1c6b12,[],be1c6b12,0,1,A problem which impairs or prevents the functi...,Bug,False,10460,AXIS,Axis,10401,Axis and Axis2 related projects,Axis,NaN,0,2005-01-02 19:13:37,2005-01-02 19:35:36,org.apache.axis.handlers.LogHandler in request...
4,29233,AXIS-1745,Decoding of service is broken in org.apache.ax...,NaN,NaN,NaN,3,Major,[],NaN,1,The issue is open and ready for the assignee t...,Open,2,new,blue-gray,To Do,NaN,97843411,[],97843411,0,1,A problem which impairs or prevents the functi...,Bug,False,10460,AXIS,Axis,10401,Axis and Axis2 related projects,Axis,NaN,1,2005-01-03 03:34:52,2005-01-03 03:34:52,The following code assumes a lot of things:\n\...



issuelinks.csv
Preview shape: (5, 36)


,key,issuelink.id,type.id,type.name,type.inward,type.outward,inwardIssue.id,inwardIssue.key,inwardIssue.summary,inwardIssue.status.id,inwardIssue.status.name,inwardIssue.statusCategory.id,inwardIssue.statusCategory.key,inwardIssue.statusCategory.colorName,inwardIssue.statusCategory.name,inwardIssue.priority.id,inwardIssue.priority.name,inwardIssue.issuetype.id,inwardIssue.issuetype.description,inwardIssue.issuetype.name,inwardIssue.issuetype.subtask,outwardIssue.id,outwardIssue.key,outwardIssue.summary,outwardIssue.status.id,outwardIssue.status.name,outwardIssue.statusCategory.id,outwardIssue.statusCategory.key,outwardIssue.statusCategory.colorName,outwardIssue.statusCategory.name,outwardIssue.priority.id,outwardIssue.priority.name,outwardIssue.issuetype.id,outwardIssue.issuetype.description,outwardIssue.issuetype.name,outwardIssue.issuetype.subtask
0,XALANC-524,11181,10020,Cloners,is cloned by,is a clone of,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23545.0,XALANC-245,substring-before and substring-after,5.0,Resolved,3.0,done,green,Done,NaN,NaN,1.0,A problem which impairs or prevents the functi...,Bug,False
1,MNG-309,12414097,12310260,Related,is related to,relates to,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12793303.0,MNG-252,allow repoclean converter to read from a repos...,6.0,Closed,3.0,done,green,Done,3.0,Major,4.0,An improvement or enhancement to an existing f...,Improvement,False
2,MNG-309,12414108,12310260,Related,is related to,relates to,12793373.0,MNG-328,figure out why commons-logging-1.0.4 won't con...,6.0,Closed,3.0,done,green,Done,2.0,Critical,1.0,A problem which impairs or prevents the functi...,Bug,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DIRSERVER-32,10493,10001,dependent,is depended upon by,depends upon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21073.0,DIRSERVER-15,Write a ReferralRule,6.0,Closed,3.0,done,green,Done,3.0,Major,1.0,A problem which impairs or prevents the functi...,Bug,False
4,DIRSERVER-32,10492,10001,dependent,is depended upon by,depends upon,17793.0,DIRSERVER-56,Create digester rules to decode all LDAPv3 mes...,6.0,Closed,3.0,done,green,Done,1.0,Blocker,1.0,A problem which impairs or prevents the functi...,Bug,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



changelog.csv
Preview shape: (5, 11)


,id,key,id_1,author,created,field,fieldtype,from,fromString,to,toString
0,13322636,BEAM-10705,19154171,b20375d7,2020-09-14 20:15:48,RemoteIssueLink,jira,212540,"This issue links to ""GitHub Pull Request #1281...",212540.0,"This issue links to ""GitHub Pull Request #1281..."
1,12757250,SQOOP-1781,14416397,8208a620,2015-03-16 19:18:18,Attachment,jira,12703176,SQOOP-1781.patch,NaN,NaN
2,12767246,PHOENIX-1580,14446744,dd305e66,2015-04-02 20:50:53,Comment,jira,So I recommend you implement explain like this...,NaN,NaN,NaN
3,12913776,FLINK-3034,15470328,b20375d7,2016-06-22 09:47:05,RemoteIssueLink,jira,43095,"This issue links to ""GitHub Pull Request #1813...",43095.0,"This issue links to ""GitHub Pull Request #1813..."
4,13213484,SPARK-26817,17444311,b20375d7,2019-02-10 17:30:20,RemoteIssueLink,jira,145292,"This issue links to ""GitHub Pull Request #2372...",145292.0,"This issue links to ""GitHub Pull Request #2372..."


In [6]:
for name, df in {
    "issues.csv": issues_preview,
    "issuelinks.csv": links_preview,
    "changelog.csv": changelog_preview,
}.items():
    schema = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_%": (df.isna().mean() * 100).round(2).values,
    })
    print()
    print("Schema:", name)
    display(schema)



Schema: issues.csv


,column,dtype,missing_%
0,id,int64,0.0
1,key,object,0.0
2,summary,object,0.0
3,resolution.id,float64,40.0
4,resolution.description,object,40.0
5,resolution.name,object,40.0
6,priority.id,int64,0.0
7,priority.name,object,0.0
8,labels,object,0.0
9,assignee,object,60.0



Schema: issuelinks.csv


,column,dtype,missing_%
0,key,object,0.0
1,issuelink.id,int64,0.0
2,type.id,int64,0.0
3,type.name,object,0.0
4,type.inward,object,0.0
5,type.outward,object,0.0
6,inwardIssue.id,float64,60.0
7,inwardIssue.key,object,60.0
8,inwardIssue.summary,object,60.0
9,inwardIssue.status.id,float64,60.0



Schema: changelog.csv


,column,dtype,missing_%
0,id,int64,0.0
1,key,object,0.0
2,id_1,int64,0.0
3,author,object,0.0
4,created,object,0.0
5,field,object,0.0
6,fieldtype,object,0.0
7,from,object,0.0
8,fromString,object,20.0
9,to,float64,40.0


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Interpretation:</b> At this point we are not changing the data. We are checking whether the real columns support the ScaleFlow requirements and confirming which fields can be safely used later.</div>

<a id="section4"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">4. Clean and Prepare the Issues Dataset</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">4.1 Cleaning rules</h3>
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Rule</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Reason</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Trim task keys and categorical text</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Prevents whitespace from creating false categories or broken joins.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Convert empty categorical values to <code>Unknown</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Keeps missing categories explicit instead of silently dropping rows.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Parse timestamps with <code>pd.to_datetime</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Creates reliable time calculations and chronological splits.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Remove duplicate task keys</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">One JIRA issue should represent one task row.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Keep unresolved issues in the final analytical dataset</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">They are valid operational records even though they do not have a supervised target yet.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Reject negative lifecycle durations</td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">A completion time earlier than creation is inconsistent data.</td></tr>
</table>
</div>

In [7]:
ISSUE_COLUMNS = [
    "key",
    "project.key",
    "priority.name",
    "issuetype.name",
    "issuetype.subtask",
    "status.name",
    "resolution.name",
    "created",
    "updated",
    "resolutiondate",
]

if CLEAN_ISSUES_PATH.exists():
    CLEAN_ISSUES_PATH.unlink()

print("Temporary clean-issues file:", CLEAN_ISSUES_PATH)


Temporary clean-issues file: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Data\processed\_clean_issues.csv


<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Data leakage rule:</b> Final status, resolution type, completion date, update span, dependency history, and changelog counts may describe the future. They can stay in the enriched analytical dataset, but they must not automatically become input features for a model that predicts at issue creation time.</div>

In [8]:
first_write = True
seen_keys = set()
rows_written = 0
rows_removed_as_duplicates = 0

reader = pd.read_csv(
    ISSUES_PATH,
    compression="zip",
    usecols=ISSUE_COLUMNS,
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

for chunk_number, chunk in enumerate(reader, start=1):
    chunk = chunk.rename(columns={
        "key": "task_key",
        "project.key": "project_key",
        "priority.name": "priority",
        "issuetype.name": "issue_type",
        "issuetype.subtask": "is_subtask",
        "status.name": "status_name",
        "resolution.name": "resolution_name",
        "created": "created_date",
        "updated": "updated_date",
        "resolutiondate": "completed_date",
    })

    chunk["task_key"] = chunk["task_key"].astype("string").str.strip()
    chunk["task_key"] = chunk["task_key"].replace("", pd.NA)

    for column in ["project_key", "priority", "issue_type", "status_name"]:
        chunk[column] = chunk[column].astype("string").str.strip()
        chunk[column] = chunk[column].replace("", pd.NA).fillna("Unknown")

    chunk["resolution_name"] = chunk["resolution_name"].astype("string").str.strip()
    chunk["resolution_name"] = chunk["resolution_name"].replace("", pd.NA).fillna("Unresolved")

    subtask_text = chunk["is_subtask"].astype("string").str.lower().str.strip()
    chunk["is_subtask"] = (
        subtask_text
        .map({"true": 1, "false": 0, "1": 1, "0": 0})
        .fillna(0)
        .astype("int8")
    )

    for column in ["created_date", "updated_date", "completed_date"]:
        chunk[column] = pd.to_datetime(chunk[column], errors="coerce")

    chunk = chunk.dropna(subset=["task_key", "created_date"])
    chunk = chunk.drop_duplicates(subset=["task_key"], keep="first")

    new_rows = ~chunk["task_key"].isin(seen_keys)
    rows_removed_as_duplicates += int((~new_rows).sum())
    chunk = chunk.loc[new_rows].copy()
    seen_keys.update(chunk["task_key"].tolist())

    chunk["resolution_time_days"] = (
        chunk["completed_date"] - chunk["created_date"]
    ).dt.total_seconds() / 86400
    chunk["update_span_days"] = (
        chunk["updated_date"] - chunk["created_date"]
    ).dt.total_seconds() / 86400

    chunk.loc[chunk["resolution_time_days"] < 0, "resolution_time_days"] = np.nan
    chunk.loc[chunk["update_span_days"] < 0, "update_span_days"] = np.nan

    chunk["is_resolved"] = chunk["completed_date"].notna().astype("int8")
    chunk["created_year"] = chunk["created_date"].dt.year
    chunk["created_month"] = chunk["created_date"].dt.month
    chunk["created_dayofweek"] = chunk["created_date"].dt.dayofweek

    chunk.to_csv(
        CLEAN_ISSUES_PATH,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S",
    )

    first_write = False
    rows_written += len(chunk)

    if chunk_number % 5 == 0:
        print(f"Issue rows prepared: {rows_written:,}")

print()
print("Finished issues cleaning.")
print("Rows written:", f"{rows_written:,}")
print("Duplicate keys removed:", f"{rows_removed_as_duplicates:,}")


Issue rows prepared: 499,715
Issue rows prepared: 986,398

Finished issues cleaning.
Rows written: 1,121,625
Duplicate keys removed: 8,216


<a id="section5"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">5. Build Dependency Features from Issue Links</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">5.1 Why use aggregation?</h3>
<p><code>issuelinks.csv</code> can contain several rows for one issue. ScaleFlow needs one task-level row, so we use a pandas <code>groupby</code> aggregation.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Many link rows for one JIRA issue
      ↓
groupby(task_key)
      ↓
dependency_count
inward_link_count
outward_link_count
blocking_relation_count
      ↓
One dependency-feature row per task</div>
</div>

In [9]:
links = pd.read_csv(
    LINKS_PATH,
    compression="zip",
    usecols=[
        "key", "type.name", "type.inward", "type.outward",
        "inwardIssue.key", "outwardIssue.key"
    ],
    low_memory=False,
)

links["task_key"] = links["key"].astype("string").str.strip()
links = links.dropna(subset=["task_key"])

relation_text = (
    links["type.name"].fillna("").astype(str) + " " +
    links["type.inward"].fillna("").astype(str) + " " +
    links["type.outward"].fillna("").astype(str)
).str.lower()

links["is_dependency"] = relation_text.str.contains("block", regex=False, na=False).astype("int8")
links["has_inward_link"] = links["inwardIssue.key"].notna().astype("int8")
links["has_outward_link"] = links["outwardIssue.key"].notna().astype("int8")

link_features = (
    links.groupby("task_key")
    .agg(
        issue_link_count=("task_key", "size"),
        dependency_count=("is_dependency", "sum"),
        inward_link_count=("has_inward_link", "sum"),
        outward_link_count=("has_outward_link", "sum"),
    )
)

print("Tasks with issue-link features:", f"{len(link_features):,}")


Tasks with issue-link features: 241,454


In [10]:
display(link_features.head())


,issue_link_count,dependency_count,inward_link_count,outward_link_count
task_key,,,,
ABDERA-236,1,0,1,0
ABDERA-267,1,0,1,0
ABDERA-276,2,0,1,1
ABDERA-294,1,0,0,1
ABDERA-428,1,0,1,0


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Interpretation:</b> The link table has now been reduced from relationship-level rows to one task-level feature row. These features are useful for bottleneck and risk analytics, but they are not automatically treated as creation-time model inputs.</div>

<a id="section6"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">6. Build Historical Features from Changelog</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">6.1 What we extract from history</h3>
<p>The changelog describes how issues changed over time. We count important field changes instead of keeping every history row.</p>
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Feature</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Meaning</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>total_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Total number of recorded field changes.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>status_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">How often the issue status changed.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>assignee_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">How often assignment changed.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>priority_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">How often priority changed.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>issue_type_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">How often issue type changed.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>project_change_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">How often the issue moved between projects.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>reopen_count</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Status transitions containing the word <code>reopen</code>.</td></tr>
</table>
</div>

In [11]:
changelog_features = None
rows_processed = 0

reader = pd.read_csv(
    CHANGELOG_PATH,
    compression="zip",
    usecols=["key", "field", "toString"],
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

for chunk_number, chunk in enumerate(reader, start=1):
    rows_processed += len(chunk)

    chunk["task_key"] = chunk["key"].astype("string").str.strip()
    chunk = chunk.dropna(subset=["task_key"])

    field = chunk["field"].fillna("").astype(str).str.strip().str.lower()
    new_value = chunk["toString"].fillna("").astype(str).str.strip().str.lower()

    temp = pd.DataFrame({
        "task_key": chunk["task_key"],
        "total_change_count": 1,
        "status_change_count": (field == "status").astype("int8"),
        "assignee_change_count": (field == "assignee").astype("int8"),
        "priority_change_count": (field == "priority").astype("int8"),
        "issue_type_change_count": field.isin(["issuetype", "issue type"]).astype("int8"),
        "project_change_count": (field == "project").astype("int8"),
        "reopen_count": (
            (field == "status")
            & new_value.str.contains("reopen", regex=False, na=False)
        ).astype("int8"),
    })

    part = temp.groupby("task_key").sum(numeric_only=True)

    if changelog_features is None:
        changelog_features = part
    else:
        changelog_features = changelog_features.add(part, fill_value=0)

    if chunk_number % 10 == 0:
        print(f"Changelog rows processed: {rows_processed:,}")

changelog_features = changelog_features.fillna(0).astype("int64")

print()
print("Tasks with changelog features:", f"{len(changelog_features):,}")


Changelog rows processed: 1,000,000
Changelog rows processed: 2,000,000
Changelog rows processed: 3,000,000
Changelog rows processed: 4,000,000
Changelog rows processed: 5,000,000
Changelog rows processed: 6,000,000
Changelog rows processed: 7,000,000
Changelog rows processed: 8,000,000
Changelog rows processed: 9,000,000

Tasks with changelog features: 823,611


In [12]:
display(changelog_features.head())


,total_change_count,status_change_count,assignee_change_count,priority_change_count,issue_type_change_count,project_change_count,reopen_count
task_key,,,,,,,
ABDERA-10,3,1,0,0,0,0,0
ABDERA-100,4,1,0,0,0,0,0
ABDERA-101,8,3,0,0,0,0,1
ABDERA-102,5,1,0,0,0,0,0
ABDERA-103,4,1,0,0,0,0,0


<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Leakage reminder:</b> Historical counts happen after issue creation. They are valuable for an operational snapshot model, but they are not used as predictors in the strict creation-time baseline splits created later in this notebook.</div>

<a id="section7"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">7. Merge the JIRA Sources</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">7.1 Merge strategy</h3>
<p>We treat <code>issues.csv</code> as the main task table. The aggregated link and changelog tables are merged onto it using <code>task_key</code>.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">issues.csv  ────────────────┐
                              │
issuelinks.csv → groupby ─────┼→ merge(task_key) → enriched issue row
                              │
changelog.csv  → groupby ─────┘</div>
<p>A <b>left join</b> is used so an issue is not removed just because it has no links or no changelog rows.</p>
</div>

In [13]:
if ENRICHED_PATH.exists():
    ENRICHED_PATH.unlink()

count_columns = [
    "issue_link_count", "dependency_count", "inward_link_count", "outward_link_count",
    "total_change_count", "status_change_count", "assignee_change_count",
    "priority_change_count", "issue_type_change_count", "project_change_count",
    "reopen_count",
]

first_write = True
rows_written = 0

reader = pd.read_csv(
    CLEAN_ISSUES_PATH,
    chunksize=CHUNK_SIZE,
    parse_dates=["created_date", "updated_date", "completed_date"],
    low_memory=False,
)

for chunk_number, chunk in enumerate(reader, start=1):
    chunk = chunk.merge(
        link_features, how="left", left_on="task_key", right_index=True
    )
    chunk = chunk.merge(
        changelog_features, how="left", left_on="task_key", right_index=True
    )

    for column in count_columns:
        chunk[column] = chunk[column].fillna(0).astype("int64")

    chunk["has_dependencies"] = (chunk["dependency_count"] > 0).astype("int8")

    chunk.to_csv(
        ENRICHED_PATH,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S",
    )

    first_write = False
    rows_written += len(chunk)

    if chunk_number % 5 == 0:
        print(f"Enriched rows written: {rows_written:,}")

print()
print("Merge complete.")
print("Rows in enriched dataset:", f"{rows_written:,}")


Enriched rows written: 500,000
Enriched rows written: 1,000,000

Merge complete.
Rows in enriched dataset: 1,121,625


In [14]:
enriched_preview = pd.read_csv(ENRICHED_PATH, nrows=5)
display(enriched_preview)


,task_key,resolution_name,priority,status_name,issue_type,is_subtask,project_key,completed_date,created_date,updated_date,resolution_time_days,update_span_days,is_resolved,created_year,created_month,created_dayofweek,issue_link_count,dependency_count,inward_link_count,outward_link_count,total_change_count,status_change_count,assignee_change_count,priority_change_count,issue_type_change_count,project_change_count,reopen_count,has_dependencies
0,WW-712,Fixed,Minor,Closed,Improvement,0,WW,2005-01-01 07:50:46,2005-01-01 07:47:50,2005-01-01 07:50:46,0.002037,0.002037,1,2005,1,5,0,0,0,0,9,1,0,0,0,0,0,0
1,XALANC-446,Fixed,Blocker,Resolved,Bug,0,XALANC,2004-12-30 05:30:36,2004-12-25 22:50:30,2005-01-01 10:20:52,4.277847,6.479421,1,2004,12,5,0,0,0,0,5,1,0,0,0,0,0,0
2,ROL-587,Cannot Reproduce,Critical,Closed,Bug,0,ROL,2005-01-02 15:21:00,2005-01-01 13:52:46,2005-01-02 15:21:00,1.061273,1.061273,1,2005,1,5,0,0,0,0,4,1,0,0,0,0,0,0
3,AXIS-1741,Unresolved,Major,Open,Bug,0,AXIS,NaN,2005-01-02 19:13:37,2005-01-02 19:35:36,NaN,0.015266,0,2005,1,6,0,0,0,0,3,0,0,0,0,0,0,0
4,AXIS-1745,Unresolved,Major,Open,Bug,0,AXIS,NaN,2005-01-03 03:34:52,2005-01-03 03:34:52,NaN,0.000000,0,2005,1,0,0,0,0,0,2,0,0,0,0,0,0,0


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Interpretation:</b> We now have one enriched record per JIRA task. The file contains core issue fields plus dependency and changelog summaries, but we still need to decide which columns are safe for prediction.</div>

<a id="section8"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">8. Feature Engineering, Targets & Leakage Control</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">8.1 Primary target</h3>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">resolution_time_days = completed_date − created_date</div>
<p>This target measures how long a resolved JIRA issue stayed open. It is an honest lifecycle target. It is <b>not</b> a contractual deadline-delay label.</p>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">8.2 Secondary risk proxy</h3>
<p>For a future classification baseline, we also create <code>long_resolution_risk</code>. The threshold is the 75th percentile of <b>training-only</b> resolution time. Validation and test data are never used to choose that threshold.</p>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">8.3 Creation-time baseline features</h3>
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Safe baseline column</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Why</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>project_key</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Project context available when the issue is created.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>priority</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Used only when the changelog shows no later priority changes.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>issue_type</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Used only when the changelog shows no issue-type changes.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>is_subtask</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Numeric structural property: 0 or 1.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><code>created_year</code>, <code>created_month</code>, <code>created_dayofweek</code></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Numeric features derived directly from creation time.</td></tr>
</table>

<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Not used as creation-time predictors:</b> completed date, final status, resolution type, update span, link/dependency counts without link timestamps, or post-creation changelog counts.</div>
</div>


In [15]:
BASELINE_FEATURES = [
    "project_key",
    "priority",
    "issue_type",
    "is_subtask",
    "created_year",
    "created_month",
    "created_dayofweek",
]

MODEL_EXPORT_COLUMNS = [
    "task_key",
    "created_date",
    *BASELINE_FEATURES,
    "resolution_time_days",
    "long_resolution_risk",
]

print("Baseline features:")
for feature in BASELINE_FEATURES:
    print("-", feature)


Baseline features:
- project_key
- priority
- issue_type
- is_subtask
- created_year
- created_month
- created_dayofweek


<div style="border-left:5px solid #0284C7;background-color:#E0F2FE;padding:11px 15px;margin:12px 0;border-radius:5px;color:#0C4A6E;"><b>Next preprocessing step:</b> <code>project_key</code>, <code>priority</code>, and <code>issue_type</code> are categorical. After the chronological split is defined, their categories will be learned from <b>Training only</b>, then the same columns will be reused for Validation and Test.</div>


<a id="section9"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">9. Create Time-Based Train / Validation / Test Splits</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">9.1 Why chronological splitting?</h3>
<p>ScaleFlow predicts future project behavior. A chronological split is therefore more realistic than randomly mixing older and newer issues.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Oldest eligible issues
      ↓
70% Training
      ↓
15% Validation
      ↓
15% Testing
      ↓
Newest eligible issues</div>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>BinX rule carried forward:</b> The test set is kept for final evaluation. Model choices and preprocessing decisions should be learned from Training and checked with Validation, not tuned on Test.</div>
</div>

In [16]:
split_info = pd.read_csv(
    ENRICHED_PATH,
    usecols=[
        "created_date",
        "resolution_time_days",
        "priority_change_count",
        "issue_type_change_count",
        "project_change_count",
    ],
    parse_dates=["created_date"],
)

eligible_mask = (
    split_info["resolution_time_days"].notna()
    & (split_info["priority_change_count"].fillna(0) == 0)
    & (split_info["issue_type_change_count"].fillna(0) == 0)
    & (split_info["project_change_count"].fillna(0) == 0)
)

eligible_dates = (
    split_info.loc[eligible_mask]
    .sort_values("created_date")
    .reset_index(drop=True)
)

n_eligible = len(eligible_dates)
if n_eligible < 3:
    raise ValueError("Not enough eligible rows to create train, validation, and test splits.")

train_end = max(1, int(n_eligible * 0.70))
validation_end = max(train_end + 1, int(n_eligible * 0.85))
validation_end = min(validation_end, n_eligible - 1)

train_cutoff = eligible_dates.loc[train_end - 1, "created_date"]
validation_cutoff = eligible_dates.loc[validation_end - 1, "created_date"]

training_targets = eligible_dates.loc[
    eligible_dates["created_date"] <= train_cutoff,
    "resolution_time_days",
]
risk_threshold_days = float(training_targets.quantile(0.75))


In [17]:
print("Eligible supervised rows:", f"{n_eligible:,}")
print("Training cutoff:", train_cutoff)
print("Validation cutoff:", validation_cutoff)
print("Training-only 75th percentile:", round(risk_threshold_days, 2), "days")


Eligible supervised rows: 841,248
Training cutoff: 2018-04-13 22:54:17
Validation cutoff: 2021-02-01 16:04:28
Training-only 75th percentile: 143.23 days


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Interpretation:</b> Both the time cutoffs and the secondary risk threshold are now defined without using future test information. This keeps the evaluation logic reproducible and leakage-aware.</div>

<a id="section10"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">10. Encode Categorical Features</h2>


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">10.1 Why do we encode?</h3>
<p>Machine-learning models need numerical input. The text categories <code>project_key</code>, <code>priority</code>, and <code>issue_type</code> are therefore converted into 0/1 dummy columns.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Training categories
        ↓
Learn allowed category values
        ↓
pd.get_dummies(...)
        ↓
Reuse exactly the same dummy columns
for Validation and Test
        ↓
Numeric ML-ready features</div>

<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Leakage control:</b> category values are learned from Training only. A category that appears later in Validation or Test is mapped to <code>Unknown</code> instead of changing the feature space.</div>
</div>


In [18]:
CATEGORICAL_FEATURES = [
    "project_key",
    "priority",
    "issue_type",
]

NUMERIC_FEATURES = [
    "is_subtask",
    "created_year",
    "created_month",
    "created_dayofweek",
]

# Store the category values found in Training only
category_values = {
    column: {"Unknown"}
    for column in CATEGORICAL_FEATURES
}

reader = pd.read_csv(
    ENRICHED_PATH,
    usecols=[
        "created_date",
        "resolution_time_days",
        "priority_change_count",
        "issue_type_change_count",
        "project_change_count",
        *CATEGORICAL_FEATURES,
    ],
    chunksize=CHUNK_SIZE,
    parse_dates=["created_date"],
    low_memory=False,
)

for chunk in reader:
    eligible = (
        chunk["resolution_time_days"].notna()
        & (chunk["priority_change_count"].fillna(0) == 0)
        & (chunk["issue_type_change_count"].fillna(0) == 0)
        & (chunk["project_change_count"].fillna(0) == 0)
        & (chunk["created_date"] <= train_cutoff)
    )

    train_chunk = chunk.loc[eligible]

    for column in CATEGORICAL_FEATURES:
        values = (
            train_chunk[column]
            .fillna("Unknown")
            .astype(str)
            .unique()
        )
        category_values[column].update(values)

# Sort categories so the output columns are reproducible
for column in CATEGORICAL_FEATURES:
    category_values[column] = sorted(category_values[column])

category_summary = pd.DataFrame({
    "feature": CATEGORICAL_FEATURES,
    "training_categories": [
        len(category_values[column])
        for column in CATEGORICAL_FEATURES
    ],
})

display(category_summary)


,feature,training_categories
0,project_key,587
1,priority,15
2,issue_type,37


In [19]:
# Build the exact dummy-column list that every split must use
DUMMY_COLUMNS = []

for column in CATEGORICAL_FEATURES:
    for value in category_values[column]:
        DUMMY_COLUMNS.append(f"{column}_{value}")

ML_FEATURE_COLUMNS = NUMERIC_FEATURES + DUMMY_COLUMNS

print("Numeric base features:", len(NUMERIC_FEATURES))
print("Dummy features:", len(DUMMY_COLUMNS))
print("Total model features:", len(ML_FEATURE_COLUMNS))


Numeric base features: 4
Dummy features: 639
Total model features: 643


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Interpretation:</b> The model feature space is now fixed using Training only. This lets Validation and Test use the same numerical columns without leaking future category information.</div>


<a id="section11"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">11. Export Final Dataset & ML-Ready Splits</h2>


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>The final export now produces two types of output:</p>
<ol style="line-height:1.8;">
<li><code>final_dataset.csv</code> — the full enriched analytical dataset with readable categorical values.</li>
<li><code>train.csv</code>, <code>validation.csv</code>, and <code>test.csv</code> — chronological supervised splits where all model features are numerical.</li>
</ol>
<div style="border-left:5px solid #0284C7;background-color:#E0F2FE;padding:11px 15px;margin:12px 0;border-radius:5px;color:#0C4A6E;"><b>Important:</b> <code>task_key</code> and <code>created_date</code> remain in the split files for traceability, but they are identifiers / audit columns, not model input features.</div>
</div>


In [20]:
for path in [FINAL_DATASET_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if path.exists():
        path.unlink()

first_final = True
first_split = {
    "train": True,
    "validation": True,
    "test": True,
}

counts = {
    "final": 0,
    "train": 0,
    "validation": 0,
    "test": 0,
}

reader = pd.read_csv(
    ENRICHED_PATH,
    chunksize=CHUNK_SIZE,
    parse_dates=["created_date"],
    low_memory=False,
)

for chunk_number, chunk in enumerate(reader, start=1):
    # Secondary target created with the Training-only threshold
    chunk["long_resolution_risk"] = pd.Series(
        pd.NA,
        index=chunk.index,
        dtype="Int64",
    )

    resolved = chunk["resolution_time_days"].notna()

    chunk.loc[resolved, "long_resolution_risk"] = (
        chunk.loc[resolved, "resolution_time_days"]
        >= risk_threshold_days
    ).astype("int8")

    # Save the readable enriched dataset
    chunk.to_csv(
        FINAL_DATASET_PATH,
        mode="w" if first_final else "a",
        header=first_final,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S",
    )

    first_final = False
    counts["final"] += len(chunk)

    # Keep only rows valid for the creation-time supervised baseline
    eligible = (
        chunk["resolution_time_days"].notna()
        & (chunk["priority_change_count"].fillna(0) == 0)
        & (chunk["issue_type_change_count"].fillna(0) == 0)
        & (chunk["project_change_count"].fillna(0) == 0)
    )

    model_data = chunk.loc[
        eligible,
        [
            "task_key",
            "created_date",
            *CATEGORICAL_FEATURES,
            *NUMERIC_FEATURES,
            "resolution_time_days",
            "long_resolution_risk",
        ],
    ].copy()

    # Chronological split
    train_part = model_data.loc[
        model_data["created_date"] <= train_cutoff
    ].copy()

    validation_part = model_data.loc[
        (model_data["created_date"] > train_cutoff)
        & (model_data["created_date"] <= validation_cutoff)
    ].copy()

    test_part = model_data.loc[
        model_data["created_date"] > validation_cutoff
    ].copy()

    split_parts = [
        ("train", train_part, TRAIN_PATH),
        ("validation", validation_part, VALIDATION_PATH),
        ("test", test_part, TEST_PATH),
    ]

    for split_name, part, path in split_parts:
        if part.empty:
            continue

        # Use only categories learned from Training
        for column in CATEGORICAL_FEATURES:
            part[column] = (
                part[column]
                .fillna("Unknown")
                .astype(str)
            )

            allowed_values = category_values[column]

            part.loc[
                ~part[column].isin(allowed_values),
                column,
            ] = "Unknown"

        # Convert categories to numerical dummy columns
        part = pd.get_dummies(
            part,
            columns=CATEGORICAL_FEATURES,
            dtype="int8",
        )

        # Add any dummy columns missing from this chunk
        for column in DUMMY_COLUMNS:
            if column not in part.columns:
                part[column] = 0

        # Keep identical feature order in Train / Validation / Test
        final_columns = [
            "task_key",
            "created_date",
            *ML_FEATURE_COLUMNS,
            "resolution_time_days",
            "long_resolution_risk",
        ]

        part = part.reindex(
            columns=final_columns,
            fill_value=0,
        )

        part.to_csv(
            path,
            mode="w" if first_split[split_name] else "a",
            header=first_split[split_name],
            index=False,
            date_format="%Y-%m-%d %H:%M:%S",
        )

        first_split[split_name] = False
        counts[split_name] += len(part)

    if chunk_number % 5 == 0:
        print(
            f"final={counts['final']:,} | "
            f"train={counts['train']:,} | "
            f"validation={counts['validation']:,} | "
            f"test={counts['test']:,}"
        )

print()
print("Export complete.")
print(counts)


C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis

final=500,000 | train=317,806 | validation=28,875 | test=30,109


C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis

final=1,000,000 | train=532,759 | validation=107,426 | test=100,268


C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  part[column] = 0
C:\Users\sadee\AppData\Local\Temp\ipykernel_23944\856212495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis


Export complete.
{'final': 1121625, 'train': 588873, 'validation': 126187, 'test': 126188}


In [21]:
output_files = [FINAL_DATASET_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH, MAPPING_PATH]
output_inventory = []

for path in output_files:
    output_inventory.append({
        "file": path.name,
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else None,
    })

display(pd.DataFrame(output_inventory))


,file,exists,size_mb
0,final_dataset.csv,True,189.19
1,train.csv,True,753.38
2,validation.csv,True,161.51
3,test.csv,True,161.54
4,jira_to_scaleflow_mapping.csv,True,0.00


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Expected result:</b> <code>final_dataset.csv</code> keeps the readable enriched data, while <code>train.csv</code>, <code>validation.csv</code>, and <code>test.csv</code> contain the same numerical model feature columns and are ready for the next modeling task.</div>


<a id="section12"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">12. Data Quality Checks</h2>


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">12.1 What we verify before delivery</h3>
<p>A dataset is not ready just because the export completed. We check file existence, row counts, target completeness, chronological order, split overlap, identical columns, and numerical model features.</p>
</div>


In [22]:
train_check = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["created_date"],
)

validation_check = pd.read_csv(
    VALIDATION_PATH,
    parse_dates=["created_date"],
)

test_check = pd.read_csv(
    TEST_PATH,
    parse_dates=["created_date"],
)

summary_df = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_check),
        "min_date": train_check["created_date"].min(),
        "max_date": train_check["created_date"].max(),
        "missing_target": train_check["resolution_time_days"].isna().sum(),
    },
    {
        "split": "validation",
        "rows": len(validation_check),
        "min_date": validation_check["created_date"].min(),
        "max_date": validation_check["created_date"].max(),
        "missing_target": validation_check["resolution_time_days"].isna().sum(),
    },
    {
        "split": "test",
        "rows": len(test_check),
        "min_date": test_check["created_date"].min(),
        "max_date": test_check["created_date"].max(),
        "missing_target": test_check["resolution_time_days"].isna().sum(),
    },
])

display(summary_df)

# Check that no task appears in two splits
train_keys = set(train_check["task_key"])
validation_keys = set(validation_check["task_key"])
test_keys = set(test_check["task_key"])

print("Train / Validation overlap:", len(train_keys & validation_keys))
print("Train / Test overlap:", len(train_keys & test_keys))
print("Validation / Test overlap:", len(validation_keys & test_keys))

# Check identical output columns
print(
    "Same columns in all splits:",
    list(train_check.columns) == list(validation_check.columns)
    == list(test_check.columns),
)

# Check that every model feature is numeric
non_numeric_features = (
    train_check[ML_FEATURE_COLUMNS]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("Non-numeric model features:", non_numeric_features)

assert not (train_keys & validation_keys)
assert not (train_keys & test_keys)
assert not (validation_keys & test_keys)

assert summary_df["missing_target"].sum() == 0

assert list(train_check.columns) == list(validation_check.columns)
assert list(train_check.columns) == list(test_check.columns)

assert len(non_numeric_features) == 0

print("Quality checks: PASS")


,split,rows,min_date,max_date,missing_target
0,train,588873,2000-10-19 09:27:07,2018-04-13 22:54:17,0
1,validation,126187,2018-04-13 22:58:08,2021-02-01 16:04:28,0
2,test,126188,2021-02-01 16:21:09,2024-11-06 14:09:26,0


Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
Same columns in all splits: True
Non-numeric model features: []
Quality checks: PASS


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;"><b>Why this is a success:</b> The supervised splits contain complete targets, no task appears in more than one split, every split has the same columns, and all model feature columns are numerical.</div>


In [23]:
metadata = {
    "task": "Prepare and Preprocess the AI/ML Dataset",
    "dataset": "Apache JIRA Issues",
    "primary_target": "resolution_time_days",
    "secondary_target": "long_resolution_risk",
    "risk_threshold_days": round(risk_threshold_days, 4),
    "train_cutoff": str(train_cutoff),
    "validation_cutoff": str(validation_cutoff),
    "split_ratio": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "training_categories": category_values,
    "ml_feature_columns": ML_FEATURE_COLUMNS,
    "split_counts": counts,
    "comments_used_in_current_model": False,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved metadata:", METADATA_PATH)

for path in [CLEAN_ISSUES_PATH, ENRICHED_PATH]:
    if path.exists():
        path.unlink()
        print("Removed temporary file:", path.name)


Saved metadata: C:\Users\sadee\OneDrive\Desktop\ScaleFlow\ScaleFlow\ScaleFlow\AI_ML\Data\processed\preprocessing_metadata.json
Removed temporary file: _clean_issues.csv
Removed temporary file: _enriched_dataset.csv


<a id="section13"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">13. Task Definition of Done</h2>


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Task requirement</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">How this notebook completes it</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Obtain and inspect the selected dataset</td><td style="padding:9px;border:1px solid #CBD5E1;">Real JIRA ZIP files, columns, types, missing values, and sample rows are inspected.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Understand columns and data types</td><td style="padding:9px;border:1px solid #CBD5E1;">Schema tables are displayed for issues, links, and changelog.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Map JIRA data to ScaleFlow</td><td style="padding:9px;border:1px solid #CBD5E1;"><code>jira_to_scaleflow_mapping.csv</code> is exported.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Clean missing / inconsistent values</td><td style="padding:9px;border:1px solid #CBD5E1;">Keys, categories, booleans, timestamps, duplicates, and invalid durations are handled.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Select relevant features</td><td style="padding:9px;border:1px solid #CBD5E1;">Creation-time baseline features are separated from historical analytics features.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Create derived features</td><td style="padding:9px;border:1px solid #CBD5E1;">Calendar, lifecycle, link/dependency, and changelog features are generated.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Define target variables</td><td style="padding:9px;border:1px solid #CBD5E1;"><code>resolution_time_days</code> is the main target; <code>long_resolution_risk</code> is a training-derived secondary target.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Prepare ML-ready data</td><td style="padding:9px;border:1px solid #CBD5E1;">Categorical features are encoded using Training-only category values; final model features are numerical.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Split Train / Validation / Test</td><td style="padding:9px;border:1px solid #CBD5E1;">Chronological 70 / 15 / 15 split logic is applied.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Verify final structure</td><td style="padding:9px;border:1px solid #CBD5E1;">No split overlap, identical columns, complete targets, and numeric model features are asserted.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Document preparation steps</td><td style="padding:9px;border:1px solid #CBD5E1;">The notebook explains each major step and writes <code>preprocessing_metadata.json</code>.</td></tr>
</table>

<div style="font-size:1.12em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:16px;border-radius:8px;margin:14px 0;color:#111827;">Raw JIRA → inspection → cleaning → aggregation → merge → target preparation → leakage control → chronological split → training-only encoding → numerical ML-ready files.</div>
</div>


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Concept</th><th style="padding:9px;border:1px solid #CBD5E1;background-color:#0F766E;color:white;text-align:left;vertical-align:top;">Meaning in ScaleFlow</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Chunk processing</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Large CSV files can be processed safely without loading all raw data into RAM.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Cleaning before preprocessing</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">We first fix missing, duplicated, malformed, and inconsistent values before model-specific transformations.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>groupby aggregation</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Many link/history rows become one meaningful feature row per task.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>merge</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Separate JIRA tables are combined through a common task key.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Feature engineering</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Raw timestamps and relationships become lifecycle, calendar, link/dependency and history features.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Target honesty</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">We use lifecycle duration instead of inventing a deadline-delay label that the source data cannot support.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Data leakage</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Future information must not be used to predict the past.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Chronological evaluation</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">Older tasks train the model; newer tasks are reserved for validation and testing.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;"><b>Reproducibility</b></td><td style="padding:9px;border:1px solid #CBD5E1;vertical-align:top;">The notebook records mappings, thresholds, split dates, and output structure for the next ML task.</td></tr>
</table>
</div>

<div style="text-align:center;margin-top:28px;color:#475569;">
<b>End of ScaleFlow — Prepare and Preprocess the AI/ML Dataset</b><br>
Apache JIRA raw data → pandas preparation → leakage-aware chronological split → training-only encoding → numerical ML-ready data
</div>
